# Detecting cyberattacks in a healthcare monitoring system

## Dataset: WUSTL-EHMS-2020

| Field | Description |
|---|---|
| **Source** | [Kaggle: aghubaish/wustl-ehms-2020-dataset](https://www.kaggle.com/datasets/aghubaish/wustl-ehms-2020-dataset), uploaded by co-author Ali Ghubaish (Washington University in St. Louis) |
| **Paper** | Hady et al., *Intrusion Detection System for Healthcare Systems Using Medical and Network Data: A Comparison Study*, IEEE Access, 2020 |
| **Size** | 16,318 samples, 43 features |
| **One row** | One network flow (a captured communication between devices in the monitoring system), together with the patient's vital-sign readings transmitted at that moment |
| **Prediction** | Whether a network flow is part of a cyberattack |
| **Target** | `Label`: 0 = Normal, 1 = Attack (the attack class is the minority, about 12.5% of rows) |

The data comes from a real-time Enhanced Healthcare Monitoring System (EHMS) testbed. Sensors on a patient send vital signs over a network to a monitoring server, while an attacker performs man-in-the-middle attacks (spoofing and data injection).

The 43 features are:
- **35 network flow metrics:** packets, bytes, rates, jitter, duration, and so on.
- **8 patient biometric values:** temperature, SpO2, pulse rate, systolic/diastolic blood pressure, heart rate, respiration rate and ST segment.

**Synthetic dataset:** `synthetic_signal.csv` is only loaded for a quick pipeline test (`is_pipeline_test = True` in the data acquisition cell). It is not needed for the real run. To create it, run `synthetic_data_generator.py` from this folder.

## Objective

Train and compare three classification methods, **KNN**, **SVM** and **Random Forest**, on the full dataset, and find the best one for detecting attacks.

### Method

- **Split:** stratified 70/30 train/test split (`random_state=42`). The test set is used only once, at the end.
- **Validation:** 5-fold stratified cross-validation on the training set. Hyperparameters and decision thresholds are tuned on this same data, so the CV scores are slightly optimistic. The test-set results are the unbiased estimate.
- **Preprocessing:** encoding and scaling are done inside scikit-learn `Pipeline`s, so they are fitted on the training folds only.
- **Metric:** F1-score of the Attack class, together with precision, recall and ROC-AUC.

### Notebook structure

1. Data acquisition and exploratory data analysis (EDA)
2. Preprocessing: drop constant columns and identifier/leakage columns
3. Train/test split
4. **v1, baseline:** grid search for each model
5. **v2, log transform + threshold tuning:** ablation of `log1p` on the network features, then a tuned decision threshold
6. **v3, wider grids:** extended hyperparameter search with threshold tuning
7. Test set evaluation: v1 vs v2 vs v3

In [ ]:
# Install kagglehub if not already installed
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

In [ ]:
####################
# Data acquisition #
####################

import os
import pandas as pd

is_pipeline_test = False

if(is_pipeline_test):
  print("Loading synthetic dataset: synthetic_signal.csv")
  df = pd.read_csv("synthetic_signal.csv")
  print(f"Successfully loaded synthetic_signal.csv with shape {df.shape}")
  display(df.head())
else:
  print("Loading real dataset aghubaish/wustl-ehms-2020-dataset")
  path = kagglehub.dataset_download("aghubaish/wustl-ehms-2020-dataset")
  print("Path to dataset files:", path)

  # Find the CSV file inside the downloaded path
  files = os.listdir(path)
  csv_files = [f for f in files if f.endswith('.csv')]
  print("Downloaded files:", files)

  # Load the CSV file into a DataFrame if found
  if csv_files:
      csv_path = os.path.join(path, csv_files[0])
      df = pd.read_csv(csv_path)
      print(f"Successfully loaded {csv_files[0]} with shape {df.shape}")
      display(df.head())
  else:
      raise Exception("No CSV file found in the downloaded dataset path.")

In [ ]:
###################################
# Exploratory data analysis (EDA) #
###################################

print("=== Dataset Info (Fields & Data Types) ===")
print(df.info())

print("\n=== Missing Values ===")
missing_vals = df.isnull().sum()
print(missing_vals[missing_vals > 0] if missing_vals.sum() > 0 else "No missing values found.")

print("\n=== Duplicate Rows ===")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(exclude=['number']).columns.tolist()
print(f"\n=== Numerical Fields ({len(numerical_cols)}) ===\n{numerical_cols}")
print(f"\n=== Categorical Fields ({len(categorical_cols)}) ===\n{categorical_cols}")

# constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
constant_cols = ['Dir', 'SrcAddr', 'DstAddr', 'Dport', 'SrcGap', 'DstGap',
                 'DIntPktAct', 'dMinPktSz', 'Trans', 'DstMac']
print(f"\n=== Constant Fields (Single Unique Value) ===\n{constant_cols}")

# potential_identifiers = [col for col in df.columns if df[col].nunique() == len(df)]
potential_identifiers = ['SrcMac', 'Attack Category', 'Sport', 'Packet_num']
print(f"\n=== Potential Unique Identifiers ===\n{potential_identifiers}")

# balance between the two target classes
target_column = 'Label'  # Typically 'Label' in WUSTL-EHMS dataset
if target_column in df.columns:
    print(f"\n=== Class Balance for '{target_column}' ===")
    class_counts = df[target_column].value_counts()
    class_percentages = df[target_column].value_counts(normalize=True) * 100
    for val, count in class_counts.items():
        print(f"Class {val}: {count} ({class_percentages[val]:.2f}%)")
else:
    print(f"\nTarget column '{target_column}' not found. Let's look for columns with 'attack' or 'label' in name:")
    possible_targets = [col for col in df.columns if 'label' in col.lower() or 'attack' in col.lower()]
    print(possible_targets)

In [ ]:
#################
# Preprocessing #
#################

# Cleanup
print(f"Dropping constant columns: {constant_cols}")
df_cleaned = df.drop(columns=constant_cols, errors='ignore')

print(f"Dropping identifiers columns: {potential_identifiers}")
df_cleaned = df_cleaned.drop(columns=potential_identifiers, errors='ignore')

# Duplicate rows would leak identical samples into both train and test
n_before = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates()
print(f"Duplicate rows removed after dropping columns: {n_before - len(df_cleaned)}")

display(df_cleaned.head())

In [ ]:
####################
# Train/test split #
####################

from sklearn.model_selection import train_test_split

# Separate features and target from df_cleaned
features = df_cleaned.drop(columns=['Label'])
target = df_cleaned['Label']

# Safety check: all features must be numeric before scaling
non_numeric = [c for c in features.select_dtypes(exclude='number').columns if c != 'Flgs']
assert not non_numeric, f"Unexpected non-numeric columns: {non_numeric}"

# Perform a stratified 70/30 training-test split
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.30, random_state=42, stratify=target
)

print("\n=== Splitting Summary ===")
print(f"Training set shape: {features_train.shape} (target: {target_train.shape})")
print(f"Test set shape (not used for fitting): {features_test.shape} (target: {target_test.shape})")
print(f"\nTraining class distribution:\n{target_train.value_counts(normalize=True).to_string()}")
print(f"\nTest class distribution:\n{target_test.value_counts(normalize=True).to_string()}")

In [ ]:
###################################
# Model training (5-fold CV) - V1 #
###################################

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

categorical_features = [col for col in categorical_cols if col in features_train.columns]
numeric_features = [col for col in numerical_cols if col in features_train.columns]

print(f"Categorical Features: {categorical_features}")
print(f"Numeric Features (scaling only): {numeric_features}\n")

# Baseline preprocessor: StandardScaler for numeric features, OneHotEncoder for categoricals (no log transform)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fitted_searches = {}

# Metrics computed during CV; refit optimizes F1
scoring_metrics = {
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc'
}

def print_best_cv_results(name, search):
    best_idx = search.best_index_
    print(f"=== {name} Best Configuration CV Performance ===")
    print(f"Best Params: {search.best_params_}")
    for metric in scoring_metrics.keys():
        mean_val = search.cv_results_[f'mean_test_{metric}'][best_idx]
        std_val = search.cv_results_[f'std_test_{metric}'][best_idx]
        print(f"{metric.upper():9}: {mean_val:.4f} ± {std_val:.4f}")
    print()

models = {
    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [1, 2, 3, 5, 7, 11, 15, 21],
         'model__weights': ['uniform', 'distance']}
    ),
    'SVM': (
        SVC(class_weight='balanced', random_state=42),
        {'model__C': [0.1, 1, 10, 100],
         'model__gamma': ['scale', 0.01, 0.1, 1]}
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced', random_state=42),
        {'model__n_estimators': [100, 300],
         'model__max_depth': [None, 10, 20],
         'model__min_samples_leaf': [1, 5, 10]}
    ),
}

for name, (estimator, param_grid) in models.items():
    print(f"--- Starting Grid Search for {name} ---")
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', estimator)])
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring_metrics,
        refit='f1',
        cv=cv_strategy,
        n_jobs=-1
    )
    search.fit(features_train, target_train)
    fitted_searches[name] = search
    print_best_cv_results(name, search)

# === Summary table ===
summary_data = []
for model_name, search in fitted_searches.items():
    best_idx = search.best_index_
    row = {'Model': model_name, 'Best Params': str(search.best_params_)}
    for metric in scoring_metrics.keys():
        row[f'mean_test_{metric}'] = search.cv_results_[f'mean_test_{metric}'][best_idx]


In [ ]:
##########################################################
# Ablation Study & Threshold Tuning on Training Set - V2 #
##########################################################

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import GridSearchCV, cross_validate, TunedThresholdClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

print(f"scikit-learn version: {sklearn.__version__} (TunedThresholdClassifierCV requires >= 1.5)\n")

# Reuses from the training cell: fitted_searches, cv_strategy, scoring_metrics,
# numeric_features, categorical_features

# --- Feature groups for the log transform ---
vital_signs = ['Temp', 'SpO2', 'Pulse_Rate', 'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST']
network_features = [c for c in numeric_features if c not in vital_signs]
vitals_to_scale = [c for c in numeric_features if c in vital_signs]

# Safety check: log1p requires non-negative values
negatives = (features_train[network_features] < 0).sum()
assert negatives.sum() == 0, f"Negative values found:\n{negatives[negatives > 0]}"
print("All network features are non-negative: log1p is safe.\n")

# Preprocessor with log1p + scaling for network features, scaling only for vitals, one-hot for Flgs
log1p_preprocessor = ColumnTransformer(
    transformers=[
        ('net', Pipeline(steps=[
            ('log1p', FunctionTransformer(np.log1p)),
            ('scaler', StandardScaler())
        ]), network_features),
        ('vital', StandardScaler(), vitals_to_scale),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# Same models and grids as in the training cell
models = {
    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [1, 2, 3, 5, 7, 11, 15, 21],
         'model__weights': ['uniform', 'distance']}
    ),
    'SVM': (
        SVC(class_weight='balanced', random_state=42),
        {'model__C': [0.1, 1, 10, 100],
         'model__gamma': ['scale', 0.01, 0.1, 1]}
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced', random_state=42),
        {'model__n_estimators': [100, 300],
         'model__max_depth': [None, 10, 20],
         'model__min_samples_leaf': [1, 5, 10]}
    ),
}

def fmt(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

def record_from_search(model_name, config_name, search):
    """Build an ablation record from a fitted GridSearchCV (best configuration)."""
    i, r = search.best_index_, search.cv_results_
    return {
        'Model': model_name,
        'Configuration': config_name,
        'Best Params': str(search.best_params_),
        'Threshold': 'Default',
        'F1-Score': fmt(r['mean_test_f1'][i], r['std_test_f1'][i]),
        'Precision': fmt(r['mean_test_precision'][i], r['std_test_precision'][i]),
        'Recall': fmt(r['mean_test_recall'][i], r['std_test_recall'][i]),
        'ROC-AUC': fmt(r['mean_test_roc_auc'][i], r['std_test_roc_auc'][i]),
        'raw_f1': r['mean_test_f1'][i],
        'estimator': search.best_estimator_,
    }

final_models = {}
final_thresholds = {}
ablation_records = []

for model_name, (estimator, param_grid) in models.items():
    print("=========================================")
    print(f" Ablation Study for: {model_name}")
    print("=========================================")

    # --- Config 1: Baseline (reused from the training cell) ---
    print("--- 1. Baseline (reused from training cell) ---")
    rec_base = record_from_search(model_name, '1. Baseline', fitted_searches[model_name])
    ablation_records.append(rec_base)

    # --- Config 2: + Log transform (new grid search, same grids and CV) ---
    print("--- 2. + Log transform (grid search) ---")
    log_pipeline = Pipeline(steps=[('preprocessor', log1p_preprocessor), ('model', estimator)])
    grid_log = GridSearchCV(
        log_pipeline, param_grid, scoring=scoring_metrics, refit='f1',
        cv=cv_strategy, n_jobs=-1
    )
    grid_log.fit(features_train, target_train)
    rec_log = record_from_search(model_name, '2. + Log Transform', grid_log)
    ablation_records.append(rec_log)

    # --- Config 3: + Tuned threshold on the better of config 1 and 2 ---
    print("--- 3. + Tuned threshold (threshold chosen inside CV folds) ---")
    better = rec_log if rec_log['raw_f1'] > rec_base['raw_f1'] else rec_base

    tuned_clf = TunedThresholdClassifierCV(better['estimator'], scoring='f1', cv=5)

    # 5-fold CV of the tuned model: the threshold is chosen inside each training fold.
    # Note: the hyperparameters were already selected on the whole training set, so these
    # CV scores are slightly optimistic. The test set gives the unbiased estimate.
    nested = cross_validate(
        tuned_clf, features_train, target_train,
        cv=cv_strategy, scoring=['f1', 'precision', 'recall'], n_jobs=-1
    )

    # Final tuned model: fit on the full training set to get the final threshold
    tuned_clf.fit(features_train, target_train)

    rec_tuned = {
        'Model': model_name,
        'Configuration': '3. + Tuned Threshold',
        'Best Params': better['Best Params'],
        'Threshold': f"{tuned_clf.best_threshold_:.4f}",
        'F1-Score': fmt(nested['test_f1'].mean(), nested['test_f1'].std()),
        'Precision': fmt(nested['test_precision'].mean(), nested['test_precision'].std()),
        'Recall': fmt(nested['test_recall'].mean(), nested['test_recall'].std()),
        'ROC-AUC': better['ROC-AUC'],   # ranking unchanged by the threshold
        'raw_f1': nested['test_f1'].mean(),
        'estimator': tuned_clf,
    }
    ablation_records.append(rec_tuned)

    # --- Select the best configuration by CV F1 ---
    best = max([rec_base, rec_log, rec_tuned], key=lambda x: x['raw_f1'])
    final_models[model_name] = best['estimator']
    if best is rec_tuned:
        final_thresholds[model_name] = tuned_clf.best_threshold_
    else:
        final_thresholds[model_name] = 0.5 if hasattr(best['estimator'], 'predict_proba') else 0.0

    print(f"-> Selected for {model_name}: {best['Configuration']} (CV F1: {best['F1-Score']})\n")

# === Summary table ===
df_ablation = pd.DataFrame(ablation_records).drop(columns=['raw_f1', 'estimator'])
print("=========================================================================")
print("                ABLATION STUDY RESULTS (5-fold CV, optimistic)")
print("=========================================================================")
display(df_ablation)

In [ ]:
###########################################################
# Extended Hyperparameter Tuning & Threshold Tuning  - V3 #
###########################################################

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import GridSearchCV, cross_validate, TunedThresholdClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Split numeric features into vitals vs network features
vital_signs = ['Temp', 'SpO2', 'Pulse_Rate', 'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST']
network_features = [c for c in numeric_features if c not in vital_signs]
vitals_to_scale = [c for c in numeric_features if c in vital_signs]

# 1. KNN Preprocessor: log1p + scale on network, scale only on vitals
knn_preprocessor = ColumnTransformer(
    transformers=[
        ('net', Pipeline(steps=[
            ('log1p', FunctionTransformer(np.log1p)),
            ('scaler', StandardScaler())
        ]), network_features),
        ('vital', StandardScaler(), vitals_to_scale),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# 2. SVM & RF Preprocessor: scale all numeric features
standard_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# Define the hyperparameter configurations
models_config = {
    'KNN': {
        'estimator': KNeighborsClassifier(),
        'preprocessor': knn_preprocessor,
        'grid': {
            'model__n_neighbors': [1, 2, 3, 5, 7, 11, 15, 21],
            'model__weights': ['uniform', 'distance'],
            'model__metric': ['euclidean', 'manhattan']
        }
    },
    'SVM': {
        'estimator': SVC(class_weight='balanced', random_state=42),
        'preprocessor': standard_preprocessor,
        'grid': {
            'model__C': [30, 100, 300, 1000],
            'model__gamma': [0.03, 0.1, 0.3]
        }
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=42),
        'preprocessor': standard_preprocessor,
        'grid': {
            'model__n_estimators': [100, 300],
            'model__max_depth': [None, 10, 20],
            'model__min_samples_leaf': [1, 5, 10],
            'model__max_features': ['sqrt', 0.3, 0.5, None],
            'model__class_weight': ['balanced', 'balanced_subsample']
        }
    }
}

final_models_ext = {}
final_thresholds_ext = {}
ext_summary_records = []

def fmt(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

for name, cfg in models_config.items():
    print(f"=========================================")
    print(f" Running v3 Pipeline & Grid Search: {name}")
    print(f"=========================================")

    # Setup pipeline with the custom preprocessor
    pipeline = Pipeline(steps=[('preprocessor', cfg['preprocessor']), ('model', cfg['estimator'])])

    # Optimize hyperparameters using CV on the training set
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=cfg['grid'],
        scoring=scoring_metrics,
        refit='f1',
        cv=cv_strategy,
        n_jobs=-1
    )
    grid_search.fit(features_train, target_train)

    best_est = grid_search.best_estimator_
    best_params = grid_search.best_params_

    # Apply threshold tuning using TunedThresholdClassifierCV
    tuned_clf = TunedThresholdClassifierCV(best_est, scoring='f1', cv=5)

    # 5-fold CV of the tuned model (threshold chosen inside each training fold).
    # Note: the hyperparameters were already selected on the whole training set, so these
    # CV scores are slightly optimistic. The test set gives the unbiased estimate.
    nested = cross_validate(
        tuned_clf, features_train, target_train,
        cv=cv_strategy,
        scoring=['f1', 'precision', 'recall', 'roc_auc'],
        n_jobs=-1
    )

    # Fit on full training set to extract final operational threshold
    tuned_clf.fit(features_train, target_train)

    final_models_ext[name] = tuned_clf
    final_thresholds_ext[name] = tuned_clf.best_threshold_

    print(f"Best Params: {best_params}")
    print(f"Final Best Threshold: {tuned_clf.best_threshold_:.4f}")
    print(f"CV F1-Score:  {fmt(nested['test_f1'].mean(), nested['test_f1'].std())}")
    print(f"CV Precision: {fmt(nested['test_precision'].mean(), nested['test_precision'].std())}")
    print(f"CV Recall:    {fmt(nested['test_recall'].mean(), nested['test_recall'].std())}")
    print(f"CV ROC-AUC:   {fmt(nested['test_roc_auc'].mean(), nested['test_roc_auc'].std())}\n")

    ext_summary_records.append({
        'Model': name,
        'Best Params': str(best_params),
        'Threshold': f"{tuned_clf.best_threshold_:.4f}",
        'F1-Score': fmt(nested['test_f1'].mean(), nested['test_f1'].std()),
        'Precision': fmt(nested['test_precision'].mean(), nested['test_precision'].std()),
        'Recall': fmt(nested['test_recall'].mean(), nested['test_recall'].std()),
        'ROC-AUC': fmt(nested['test_roc_auc'].mean(), nested['test_roc_auc'].std())
    })

# Construct summary table
df_ext_summary = pd.DataFrame(ext_summary_records)
print("=========================================================================")
print("          SUMMARY OF EXTENDED TUNING RESULTS (5-fold CV, optimistic)")
print("=========================================================================")
display(df_ext_summary)

In [ ]:
#############################################
# Evaluation and comparison: v1 vs v2 vs v3 #
#############################################

# The test set is only touched here, once per final model.
#   v1: baseline grid search (training cell), default threshold
#   v2: log transform + tuned threshold (best config from the ablation study)
#   v3: wider grids + tuned threshold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score
)

# Each version: (fitted models, thresholds). None = default threshold (0.5 for
# probabilities, 0.0 for SVM decision-function scores).
versions = {
    'v1 (baseline)': (
        {name: search.best_estimator_ for name, search in fitted_searches.items()},
        None
    ),
    'v2 (log + threshold)': (final_models, final_thresholds),
    'v3 (wider grids)': (final_models_ext, final_thresholds_ext),
}

test_results = []

for version, (version_models, version_thresholds) in versions.items():
    for name, model in version_models.items():
        print("=========================================================")
        print(f" {version} - {name}: Test Set Evaluation")
        print("=========================================================")

        # Get scores/probabilities on the test partition
        if hasattr(model, "predict_proba"):
            test_scores = model.predict_proba(features_test)[:, 1]
            threshold = 0.5
        else:
            test_scores = model.decision_function(features_test)
            threshold = 0.0

        if version_thresholds is not None:
            threshold = version_thresholds[name]

        if version_thresholds is None:
            # Untuned model: use predict() exactly as the standalone v1 evaluation does
            # (ties at exactly 0.5, e.g. KNN with even k, go to class 0)
            preds = model.predict(features_test)
        else:
            preds = (test_scores >= threshold).astype(int)

        print(f"\n--- [Threshold: {threshold:.4f}] Classification Report ---")
        print(classification_report(target_test, preds, target_names=['Normal (0)', 'Attack (1)']))
        print("Confusion Matrix:")
        print(confusion_matrix(target_test, preds))
        print()

        # Comparison metrics for the positive/Attack class
        prec, rec, f1, _ = precision_recall_fscore_support(
            target_test, preds, average='binary', pos_label=1
        )
        test_results.append({
            'Version': version,
            'Model': name,
            'Threshold': threshold,
            'Precision (Attack)': prec,
            'Recall (Attack)': rec,
            'F1-Score (Attack)': f1,
            'ROC-AUC': roc_auc_score(target_test, test_scores)
        })

# === Comparison tables ===
df_test_summary = pd.DataFrame(test_results)
print("=========================================================================")
print("            TEST SET PERFORMANCE: v1 vs v2 vs v3 (all models)")
print("=========================================================================")
display(df_test_summary.round(4))

print("\nTest F1-Score (Attack) by model and version:")
display(
    df_test_summary
    .pivot(index='Model', columns='Version', values='F1-Score (Attack)')
    [list(versions)]
    .round(4)
)